In [ ]:
import numpy as np
from pathlib import Path
base = Path('./mnist_dataset')

def get_imgs(path):
    with open(path, 'rb') as f:
        imgs_raw = f.read()
    n = int.from_bytes(imgs_raw[4:8])
    imgs = np.frombuffer(imgs_raw, dtype=np.uint8, offset=16).reshape(n, 28, 28)
    return imgs

def get_labels(path):
    with open(path, 'rb') as f:
        labels_raw = f.read()
    return np.frombuffer(labels_raw, dtype=np.uint8, offset=8)

img_size = 28**2
train_img_path = base / 'train-images.idx3-ubyte'
train_label_path = base / 'train-labels.idx1-ubyte'
test_img_path = base / 't10k-images.idx3-ubyte'
test_label_path = base / 't10k-labels.idx1-ubyte'
train_images = get_imgs(train_img_path)
test_images = get_imgs(test_img_path)
train_labels = get_labels(train_label_path).tolist()
test_labels = get_labels(test_label_path).tolist()

print(f"{len(train_images)} images; {len(train_labels)} labels")


In [ ]:
def softmax(vec):
    tmp = np.exp(vec)
    return tmp / np.sum(tmp)

relu = np.vectorize(lambda x: max(0, x))

def cost(img: np.ndarray, label: int, net):
    pred = net.run(img)
    return -np.log(pred[label])

def avg_cost(net):
    tc = 0
    for img, label in zip(train_images, train_labels):
        tc += cost(img, label, net)
    return tc / len(train_images)

def to_one_hot(label, n=10):
    one_hot = np.zeros(n)
    one_hot[label] = 1
    return one_hot

In [ ]:
class Network:
    def __init__(self, layer_sizes=[28**2, 16, 16, 10], lr=0.01):
        self.n_layers = len(layer_sizes)
        self.biases = [np.random.rand(layer_sizes[i])-0.5 for i in range(1, len(layer_sizes))]
        self.weights = [np.random.rand(layer_sizes[i],layer_sizes[i-1])-0.5 for i in range(1, len(layer_sizes))]
        self.lr = lr

    def run(self, img):
        activation = img.flatten() / 256
        for i in range(len(self.biases) - 1):
            activation = relu((self.weights[i] @ activation) + self.biases[i])
    
        return softmax((self.weights[-1] @ activation) + self.biases[-1])

    def backprop(self, img, label: np.ndarray, multiplier=1.):
        activations = []    # will be n_layers - 1 long, last layer isn't activated just softmax'd
        logits = []     # n_layers long

        flattened = img.flatten() / 256
        activations.append(flattened)
        logits.append(flattened)
        for i in range(self.n_layers - 1):
            logits.append((self.weights[i] @ activations[-1]) + self.biases[i])
            activations.append(relu(logits[-1]))    # the last activation is useless here
        d_activations = [(z >= 0).astype(int) for z in logits]
        pred = softmax(logits[-1] - max(logits[-1]))

        activations = [a.reshape(1, len(a)) for a in activations]
        delta_l = (pred - label)
        for i in range(self.n_layers - 2, -1, -1):
            old_weight = self.weights[i].copy()
            self.biases[i] -= self.lr * delta_l * multiplier
            self.weights[i] -= self.lr * delta_l.reshape(-1, 1) * activations[i] * multiplier
            delta_l = np.multiply((delta_l @ old_weight).flatten(), d_activations[i])

    def train(self, images, labels: list[int], epochs):
        encoded_labels = [to_one_hot(l) for l in labels]
        for iteration in range(epochs):
            print(f"Iteration {iteration+1}")
            print(f"Average cost: {avg_cost(self)}")
            for img, label in zip(images, encoded_labels):
                self.backprop(img, label, 1/(1+iteration))


net = Network()
net.train(train_images, train_labels, 10)


In [ ]:
correct = 0
for i in range(len(test_images)):
    pred = net.run(test_images[i])
    correct += int(np.argmax(pred) == test_labels[i])

correct / len(test_images)